# Step 5 : backward passによる入力修正則の導出

## 概要

Step4で導出したQ関数の二次近似から、入力修正則を導出し、それを用いた価値関数の更新から、backward passの計算を導出する。

## 1. 入力修正則の導出

Step4ではQ関数を基準軌道の近傍で次のように二次近似した。

$$
\begin{aligned}
Q_n(\bar{X}_n + \delta X_n, \bar{u}_n+ \delta u_n) =& \ell_n(\bar{X}_n + \delta X_n, \bar{u}_n+ \delta u_n) + V_{n+1}(\bar{X}_{n+1} + \delta X_{n+1}) \\
\approx & \bar{Q}_n + Q_{X,n}^T \delta X_n + Q_{u,n}^T \delta u_n \\
& + \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n + \delta u_n^T Q_{uX,n} \delta X_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n
\end{aligned}
$$

$$
\begin{aligned}
\bar{Q}_n =& \ell_n(\bar{X}_n, \bar{u}_n) + \bar{V}_{n+1} \\
Q_{X,n} =& \ell_{X,n} + A_n^T V_{X,n+1} \\
Q_{u,n} =& \ell_{u,n} + B_n^T V_{X,n+1} \\
Q_{XX,n} =& \ell_{XX,n} + A_n^T V_{XX, n+1} A_n \\
Q_{uX,n} =& \ell_{uX,n} + B_n^T V_{XX, n+1} A_n \\
Q_{uu,n} =& \ell_{uu,n} + B_n^T V_{XX, n+1} B_n
\end{aligned}
$$


この$Q_n$を$\delta u_n$で偏微分してゼロと置き、$\delta u_n$を求めることで、それが時刻$n$における入力修正則になることを示す。

まず、$\delta u_n$で偏微分を行うため、$\delta u_n$に関する項でまとめると次のようになる。$C$は$\delta u_n$に依存しない項である。

$$
Q_n(\delta u_n) = (Q_{u,n} + Q_{uX,n} \ \delta X_n)^T \delta u_n + \frac{1}{2}\delta u_n^T \ Q_{uu,n} \ \delta u_n + C 
$$

この$Q_n(\delta u_n)$を$\delta u_n$で偏微分すると、以下となる。

$$
\frac{\partial Q_n}{\partial \delta u_n} = Q_{u,n} + Q_{uX,n} \ \delta X_n + Q_{uu,n} \ \delta u_n
$$

最小点の候補は以下のように、一階偏微分の傾きがゼロになる停留条件を満たす$\delta u_n$である。

$$
\begin{aligned}
\frac{\partial Q_n}{\partial \delta u_n} &= Q_{u,n} + Q_{uX,n} \ \delta X_n + Q_{uu,n} \ \delta u_n = 0 \\
& \downarrow \\
\delta u_n &= -Q_{uu,n}^{-1} \ Q_{u,n} - Q_{uu,n}^{-1} \ Q_{uX,n} \ \delta X_n 
\end{aligned}
$$

そのような$\delta u_n$を入力修正則 $\delta u_n^\star$とする。

$$
\boxed{
\delta u_n^\star =  k_n + K_n\ \delta X_n 
}
$$

$$
\boxed{
k_n = -Q_{uu,n}^{-1} \ Q_{u,n}, \quad K_n = - Q_{uu,n}^{-1} \ Q_{uX,n}
}
$$

この入力修正則は、状態摂動$\delta X_{n}$に対して、時刻$n$のステージコストと、時刻$n+1$から終端までの将来コストを合わせた局所二次Q関数が最小になるように、時刻$n$の入力をどれだけ修正するかを計算する。

入力修正則により、以下の２項目の和である局所二次Q関数を最小化する。

$$
Q_n = \underbrace{\ell_n}_{\text{時刻}n\text{のコスト}} + \underbrace{V_{n+1}}_{\text{時刻}n+1\text{以降の最小コスト}}
$$

backward passでは、$\delta X_n$に具体的な値を与えるのではなく、$\delta X_n$を変数として残したまま、任意の状態摂動に対する入力修正則を構成する$k_n, K_n$を計算する。


この入力修正則が、時刻$n$のステージコストと時刻$n+1$以降の価値関数から構成された局所二次Q関数を最小化することを確認する。

それには、$Q_n$の$\delta u_n$によるHessianが正定値であることを確認すればよい。

つまり、次のように$Q_{uu,n}$が正定値であることが、一意な最小点となるための十分条件である。この場合、$Q_n$は$\delta u_n$に関して狭義凸関数となり、一意な最小点が求まる。

$$
\frac{\partial^2 Q_n}{\partial \delta u_n^2} = Q_{uu,n} \succ 0
$$

$Q_{uu,n}$は以下で構成される。

$$
Q_{uu,n} = \ell_{uu,n} + B_n^T V_{XX, n+1} B_n
$$

よって、

$$
\ell_{uu,n} \succ 0 , \quad V_{XX, n+1} \succeq 0
$$

なら半正定値行列の性質より、以下となる。

$$
B_n^T V_{XX, n+1} B_n \succeq 0
$$

よって、$Q_{uu,n}$が正定値行列であることが保証される。

$$
Q_{uu,n} \succ 0
$$

しかし、非線形で非凸な問題では、局所二次近似によって得られる$V_{XX,n+1}$やコストのHessianが不定になることがあり、$Q_{uu,n}$が正定値であることは必ずしも保証されない。この問題は Step8 の数値安定化で扱う。このNoteでは$Q_{uu,n}$が正定値であると仮定する。


## 2. 価値関数の更新式

上記の入力修正則の係数($k_n, K_n$)は($Q_{uu,n},Q_{u,n},Q_{uX,n}$)で構成されており、($Q_{uu,n},Q_{u,n},Q_{uX,n}$)は価値関数($V_{XX,n+1}, V_{X, n+1}$)から構成されている。よって以下の流れがある。

$$
(V_{XX,n+1}, V_{X, n+1}) \rightarrow (Q_{uu,n},Q_{u,n},Q_{uX,n}) \rightarrow (k_n, K_n)
$$

次は($k_n,K_n$)により、($V_{XX,n}, V_{X, n}$)が求まることを示す。

入力修正則 $\delta u_n^\star = k_n + K_n \delta X_n$ を局所二次Q関数 $\tilde{Q}_n$に代入する。$\tilde{Q}_n$は$\delta u_n^\star$により最小化されているため、次のように価値関数$V_n$の二次近似$\tilde{V}_n$と等式として構成することが出来る。

$$
\begin{aligned}
\tilde{V}_n(\bar{X}_n + \delta X_n) &= \underset{\delta u_n}{\min} \tilde{Q}_n(\bar{X}_n + \delta X_n , \bar{u}_n + \delta u_n) \\
\tilde{V}_n(\bar{X}_n + \delta X_n) &= \tilde{Q}_n(\bar{X}_n + \delta X_n , \bar{u}_n + \delta u_n^\star)
\end{aligned}
$$

左辺の価値関数の二次近似式$\tilde{V}_n$は以下である。

$$
\tilde{V}_n(\bar{X}_n + \delta X_n) = \bar{V}_{n} + V_{X, n}^T \delta X_{n} + \frac{1}{2} \delta X_{n}^T \ V_{XX, n} \ \delta X_{n}
$$

右辺のQ関数の二次近似式$\tilde{Q}_n$は以下である。

$$
\begin{aligned}
\tilde{Q}_n(\bar{X}_n + \delta X_n, \bar{u}_n+ \delta u_n^\star) 
= & \bar{Q}_n  + \left( Q_{u,n} + Q_{uX,n} \delta X_n \right)^T \delta u_n^\star + \frac{1}{2} (\delta {u_n^{\star}})^T Q_{uu,n} \delta u_n^\star \\
& + Q_{X,n}^T \delta X_n + \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n 
\end{aligned}
$$

$\delta u_n^\star$に関する項目だけを展開すると、以下のようにまとまる。

$$
\begin{aligned}
 & \left( Q_{u,n} + Q_{uX,n} \delta X_n \right)^T (k_n + K_n \delta X_n) + \frac{1}{2} (k_n + K_n \delta X_n)^T Q_{uu,n} (k_n + K_n \delta X_n) \\
 &= Q_{u,n}^T \ k_n + Q_{u,n}^T \ K_n \delta X_n + k_n^T Q_{uX,n} \delta X_n + \delta X_n ^T Q_{uX,n}^T K_n \delta X_n \\
 & \space \space +  \frac{1}{2} k_n^T Q_{uu,n} k_n + \frac{1}{2} k_n^T Q_{uu,n}  K_n \delta X_n + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} k_n + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} K_n \delta X_n \\
 &= Q_{u,n}^T \ k_n + \frac{1}{2} k_n^T Q_{uu,n} k_n \\
 & \space \space + \left( K_n^T Q_{u,n} \   + Q_{uX,n}^T k_n + K_n^T  Q_{uu,n} k_n\right)^T \delta X_n \\
 & \space \space + \delta X_n^T Q_{uX,n}^T K_n \delta X_n  + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} K_n \delta X_n
\end{aligned}
$$

２行目の式から３行目の式では、以下の２項はともにスカラーであり、$Q_{uu,n}$は対称なので、一つにまとめられる関係を用いている。

$$
\frac{1}{2} k_n^T Q_{uu,n}  K_n \delta X_n + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} k_n
$$


二次近似式 $\tilde{Q}_n$の項で、定数項、$\delta X_n$の一次項、二次項でまとめると次のようになる。

- 定数項

$$
\bar{Q}_n + Q_{u,n}^T \ k_n + \frac{1}{2} k_n^T Q_{uu,n} k_n
$$

- 一次項

$$

Q_{X,n}^T \delta X_n + \left( K_n^T Q_{u,n} \   + Q_{uX,n}^T k_n + K_n^T  Q_{uu,n} k_n  \right)^T \delta X_n \\
= \left(Q_{X,n} + K_n^T Q_{u,n} \   + Q_{uX,n}^T k_n + K_n^T  Q_{uu,n} k_n \right)^T \delta X_n
$$

- 二次項

$$
\begin{aligned}
& \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n  + \delta X_n^T Q_{uX,n}^T K_n \delta X_n  + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} K_n \delta X_n \\
& = \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n  + \frac{1}{2} \delta X_n^T (Q_{uX,n}^T K_n + K_n^T Q_{uX,n} )\delta X_n  + \frac{1}{2} \delta X_n^T K_n^T  Q_{uu,n} K_n \delta X_n \\
& = \frac{1}{2} \delta X_n^T \left(Q_{XX,n} + Q_{uX,n}^T K_n + K_n^T Q_{uX,n} + K_n^T  Q_{uu,n} K_n \right) \delta X_n 
\end{aligned}
$$

$k_n, K_n$の定義から得られる次の関係を用いて、各項を整理する。

$$
Q_{uu,n} k_n = - Q_{u,n}, \quad Q_{uu,n} K_n = - Q_{uX,n}
$$

- 定数項

$$
\bar{Q}_n + Q_{u,n}^T \ k_n - \frac{1}{2} k_n^T Q_{u,n} = \bar{Q}_n + \frac{1}{2} Q_{u,n}^T \ k_n
$$

- 一次項

$$
\begin{aligned}
&\left(Q_{X,n} + K_n^T Q_{u,n}  + Q_{uX,n}^T k_n + K_n^T  Q_{uu,n} k_n \right)^T \delta X_n \\
&= \left(Q_{X,n} + \cancel{K_n^T Q_{u,n}}  + Q_{uX,n}^T k_n - \cancel{K_n^T Q_{u,n} }\right)^T \delta X_n \\
&= \left(Q_{X,n} + Q_{uX,n}^T k_n \right)^T \delta X_n
\end{aligned}
$$

- 二次項

$$
\begin{aligned}
& \frac{1}{2} \delta X_n^T \left(Q_{XX,n} + Q_{uX,n}^T K_n + K_n^T Q_{uX,n} + K_n^T  Q_{uu,n} K_n \right) \delta X_n \\
& = \frac{1}{2} \delta X_n^T \left(Q_{XX,n} + Q_{uX,n}^T K_n + \cancel{K_n^T Q_{uX,n}} - \cancel{K_n^T Q_{uX,n}} \right) \delta X_n \\
& = \frac{1}{2} \delta X_n^T \left(Q_{XX,n} + Q_{uX,n}^T K_n \right) \delta X_n
\end{aligned}
$$


これより、$\tilde{Q}_n(\bar{X}_n + \delta X_n , \bar{u}_n + \delta u_n^\star)$は以下のように構成できる。

$$
\begin{aligned}
\tilde{Q}_n(\bar{X}_n + \delta X_n , \bar{u}_n + \delta u_n^\star) =& \left(\bar{Q}_n + \frac{1}{2} Q_{u,n}^T \ k_n \right) +  \left(Q_{X,n} + Q_{uX,n}^T k_n \right)^T \delta X_n \\ 
&+ \frac{1}{2} \delta X_n^T \left(Q_{XX,n} + Q_{uX,n}^T K_n \right) \delta X_n
\end{aligned}
$$

よって、($k_n, K_n$)を用いて、価値関数 $\tilde{V}_n$の項は以下のようになる。このうち必要なのは、$k_{n-1},K_{n-1}$の構成に必要な$Q_{uu,n-1},Q_{uX,n-1},Q_{u,n-1}$であり、これらの構成に用いられている、$V_{X,n}, V_{XX,n}$である。

$$
\bar{V}_n = \bar{Q}_n + \frac{1}{2} Q_{u,n}^T \ k_n \\
$$

$$
\boxed{
V_{X,n} = Q_{X,n} + Q_{uX,n}^T k_n \\
}
$$

$$
\boxed{
V_{XX,n} = Q_{XX,n} + Q_{uX,n}^T K_n
}
$$

以上より、以下のようにbackward pass を構成することが出来る。$V_{X,n}$は$Q_{X,n}$を用いて、$V_{XX,n}$は$Q_{XX,n}$を用いて更新されているため、$Q$の項目に追加している。

$$
\begin{aligned}
(V_{XX,n+1}, V_{X, n+1}) \rightarrow& (Q_{X,n},Q_{uu,n},Q_{XX,n},Q_{u,n},Q_{uX,n}) \rightarrow (k_n, K_n)\\
\rightarrow (V_{XX,n}, V_{X, n}) \rightarrow& (Q_{X,n-1},Q_{uu,n-1},Q_{XX,n-1},Q_{u,n-1},Q_{uX,n-1}) \rightarrow (k_{n-1}, K_{n-1})  \rightarrow \\
 \vdots & \\ 
\rightarrow (V_{XX,1}, V_{X, 1}) \rightarrow& (Q_{X,0},Q_{uu,0},Q_{XX,0},Q_{u,0},Q_{uX,0}) \rightarrow (k_{0}, K_{0})
\end{aligned}
$$


## 3. backward pass の計算

終端時刻$N$では入力およびステージコストが存在しないため、価値関数は終端コストと一致する。

$$
V_N(X_N) = \phi(X_N)
$$

終端での左辺の価値関数の二次近似式は以下となる。

$$
V_{N}(\bar{X}_{N} + \delta X_{N}) \approx \bar{V}_{N} + V_{X, N}^T \delta X_{N} + \frac{1}{2} \delta X_{N}^T \ V_{XX, N} \ \delta X_{N}
$$

右辺の終端コストの二次近似式は以下となる。

$$
\phi(\bar{X}_N + \delta X_N) \approx \phi(\bar{X}_N) + \phi_X^T \ \delta X_N + \frac{1}{2} \delta X_N^T \ \phi_{XX} \ \delta X_N
$$

よって価値関数を対応させると以下となる。

$$
V_{X,N} = \phi_X, \quad V_{XX, N} = \phi_{XX}
$$

よって、終端から次のように更新をしていけば、入力修正項の$k_n, K_n$の系列を求めることが出来る。

$$
\begin{aligned}
(V_{XX,N}, V_{X, N}) \rightarrow& (Q_{X,N-1},Q_{u,N-1},Q_{uu,N-1},Q_{uX,N-1},Q_{XX,N-1}) \rightarrow (k_{N-1}, K_{N-1}) \rightarrow \\
 & \vdots \\
(V_{XX,n+1}, V_{X, n+1}) \rightarrow& (Q_{X,n},Q_{u,n},Q_{uu,n},Q_{uX,n},Q_{XX,n}) \rightarrow (k_{n}, K_{n})  \rightarrow \\
 \vdots & \\ (V_{XX,1}, V_{X, 1}) \rightarrow& (Q_{X,0},Q_{u,0},Q_{uu,0},Q_{uX,0},Q_{XX,0}) \rightarrow (k_{0}, K_{0})
\end{aligned}
$$


### backward pass の計算式

#### 終端 $N$

- 価値関数の係数

$$
\begin{aligned}
V_{X,N} &= \phi_X \\
V_{XX, N} &= \phi_{XX}
\end{aligned}
$$

- Q関数の係数

$$
\begin{aligned}
Q_{X,N-1} &= \ell_{X,N-1} + A_{N-1}^T V_{X,N} \\
Q_{u,N-1} &= \ell_{u,N-1} + B_{N-1}^T V_{X,N} \\
Q_{uu,N-1} &= \ell_{uu,N-1} + B_{N-1}^T V_{XX,N} B_{N-1} \\
Q_{uX,N-1} &= \ell_{uX, N-1} + B_{N-1}^T V_{XX,N} A_{N-1} \\
Q_{XX, N-1} &= \ell_{XX,N-1} + A_{N-1}^T V_{XX,N} A_{N-1} \\
\end{aligned}
$$

- 入力修正項の係数

$$
\begin{aligned}
k_{N-1} &= -Q_{uu,N-1}^{-1} \ Q_{u,N-1} \\
 K_{N-1} &= - Q_{uu,N-1}^{-1} \ Q_{uX,N-1}
\end{aligned}
$$

#### 時刻 $n$

- 価値関数の係数

$$
\begin{aligned}
V_{X,n+1} &= Q_{X, n+1} + Q_{uX, n+1}^T k_{n+1} \\
V_{XX, n+1} &= Q_{XX, n+1} + Q_{uX, n+1}^T K_{n+1}
\end{aligned}
$$

- Q関数の係数

$$
\begin{aligned}
Q_{X,n} &= \ell_{X,n} + A_{n}^T V_{X,n+1} \\
Q_{u,n} &= \ell_{u,n} + B_{n}^T V_{X,n+1} \\
Q_{uu,n} &= \ell_{uu,n} + B_{n}^T V_{XX,n+1} B_{n} \\
Q_{uX,n} &= \ell_{uX,n} + B_{n}^T V_{XX,n+1} A_{n} \\
Q_{XX,n} &= \ell_{XX,n} + A_{n}^T V_{XX,n+1} A_{n} \\
\end{aligned}
$$

- 入力修正項の係数

$$
\begin{aligned}
k_{n} &= -Q_{uu,n}^{-1} \ Q_{u,n} \\
 K_{n} &= - Q_{uu,n}^{-1} \ Q_{uX,n}
\end{aligned}
$$




## 4. 入力修正則についての補足

backward pass において、目標値と基準状態の差、および基準状態と候補状態の差が、それぞれどのように入力修正則に組み込まれるのかを説明する。

まず、目標値と基準状態は次のようにステージコストに入っているとする。ここで$W_X, R$はそれぞれ状態と入力に対する重みである。

$$
\ell_n (X_n, u_n) = \frac{1}{2}(X_n - X_{ref,n})^T W_X (X_n - X_{ref,n}) + \frac{1}{2} u_n^T \ R \ u_n
$$

ステージコストの状態に関する勾配は、基準状態と目標値とのズレが考慮された値になる。

$$
\ell_{X,n} = W_X(\bar{X}_n - X_{ref,n})
$$

時刻を$n+1$とした場合、この$\ell_{X,n+1}$は$Q_{X,n+1}$で以下のように組み込まれる。

$$
Q_{X,n+1} = \ell_{X,n+1} + A_{n+1}^T V_{X,n+2}
$$

そして、価値関数 $V_{X,n+1}$は次のように$n+1$の$Q$の係数と入力修正則の係数を用いて次のようになる。

$$
V_{X, n+1} = Q_{X, n+1} + Q_{uX, n+1}^T k_{n+1}
$$

この$V_{X,n+1}$は$B_n$を介して$Q_{u,n}$に伝わる。

$$
Q_{u,n} = \ell_{u,n} + B_n^T V_{X,n+1}
$$

そして、$Q_{u,n}$により、入力修正項の係数$k_n$へ伝わる。

$$
k_n = - Q_{uu,n}^{-1} Q_{u,n}
$$

よって、目標状態と基準状態との差は、次の流れで$k_n$に反映される。

$$
\bar{X}_{n+1} - X_{ref,n+1} \rightarrow \ell_{X,n+1} \rightarrow Q_{X,n+1} \rightarrow V_{X, n+1} \rightarrow Q_{u,n} \rightarrow k_n
$$

$K_n$は$Q_{XX}, V_{XX}$を介して状態に関するコストの二階微分、すなわち曲率の影響を受ける。一般に非二次コストでは、この二階微分も基準状態と目標状態の差に依存するため、その誤差情報が$K_n$に伝わる場合がある。

一方、今回のステージコストの設定では以下のように重み$W_X$のみとなる。

$$
\ell_{XX,n+1} = W_X
$$

そのため、目標値と基準状態の差は$k_n$には直接反映されるが、$K_n$には直接反映されず、状態重み$W_X$による曲率のみが反映される。

$\delta X_n$ は、forward pass において入力修正則を用いて生成された候補状態$X_n^{new}$と、現在の基準状態 $\bar{X}_n$ との差を表す。

そして、入力修正則は下記のようになっているため、

$$
\delta u_n^\star = k_n + K_n \delta X_n
$$

入力修正則の各項目には、目標状態と基準状態との差、および候補状態と基準状態との差に関する情報が、次のように組み込まれる。

- $k_n$ : コストの一階微分を通じて、目標状態と基準状態の差を考慮する
- $K_n$ : コストの二階微分が目標状態と基準状態の差に依存する場合、その差によって変化する曲率情報を考慮する
- $K_n \delta X_n$ : 候補状態と基準状態との差に応じて、$K_n$を介して入力を調整する。



## 5. backward pass における計算上の工夫

#### 逆行列を直接計算しない

入力修正則の係数は理論上は次のように表される。

$$
\begin{aligned}
k_n &= -Q_{uu,n}^{-1} Q_{u,n} \\
K_n &= -Q_{uu,n}^{-1} Q_{uX, n}
\end{aligned}
$$

しかし、実装では逆行列$Q_{uu,n}$を直接計算しない。代わりに以下の連立方程式を解く。

$$
\begin{aligned}
Q_{uu,n}k_n &= - Q_{u,n} \\
Q_{uu,n} K_n &= -Q_{uX, n}
\end{aligned}
$$

逆行列を求めて行列の積を求める方法よりも、連立方程式として解くほうが、計算量と数値誤差の面で有利である。

numpyを用いる場合は次のように計算する。

```python
k_n = np.linalg.solve(Q_uu_n, -Q_u_n)
K_n = np.linalg.solve(Q_uu_n, -Q_uX_n)
```

また、$k_n, K_n$は同じ係数行列$Q_{uu,n}$を用いるため、右辺をまとめて一度に解くこともできる。

```python
rhs = -np.column_stack((Q_u_n, Q_uX_n))
solution = np.linalg.solve(Q_uu_n, rhs)

k_n = solution[:, 0]
K_n = solution[:, 1:]
```

#### Hessianを対称化する

理論上、Hessianである$Q_{uu,n}$と$V_{XX,n}$は対称行列である。しかし、浮動小数点による丸め誤差などによって、非対角項がわずかに異なる場合がある。


そこで、行列とその転置の平均を取り、数値計算上の対称性を保つ。

$$
\begin{aligned}
Q_{uu,n} \leftarrow \frac{1}{2}(Q_{uu,n} + Q_{uu,n}^T) \\
V_{XX,n} \leftarrow \frac{1}{2}(V_{XX,n} + V_{XX,n}^T) \\
\end{aligned}
$$

これは次のような操作を行っている。

対称行列である$M$が数値計算の誤差により次のような非対称行列となってしまったとする。

$$
M = \begin{bmatrix}
a & b \\ c & d 
\end{bmatrix} , \quad M \neq M^T
$$

これを対称化すると、対応する非対角成分は両者の平均値に置き換えられる。

$$
M \leftarrow \frac{1}{2}\left(\begin{bmatrix}
a & b \\ c & d 
\end{bmatrix} + \begin{bmatrix}
a & c \\ b & d 
\end{bmatrix} \right) = \begin{bmatrix}
a & (b+c)/2 \\ (b+c)/2 & d 
\end{bmatrix}
$$

この処理は、対応する非対角成分を平均化して対称構造を回復させるものである。$Q_{uu,n}$を下記のCholesky分解へ渡す前や、$V_{XX,n}$を次の時刻へ伝播させる前に行う。ただし正定値や条件数を改善する処理ではない。

pythonによる実装では次のように行う。

```python
Q_uu_n = 0.5 * (Q_uu_n + Q_uu_n.T)

V_XX_n = Q_XX_n + Q_uX_n.T @ K_n
V_XX_n = 0.5 * (V_XX_n + V_XX_n.T)
```

$Q_{XX,n}$もHessianだが、その後に構成する$V_{XX,n}$を対称化する場合、明示的な対称化は省略できる。

$$
V_{XX,n} = Q_{XX,n} + Q_{uX,n}^T K_n
$$

#### Cholesky 分解による正定値性の確認

入力修正則が局所二次Q関数の一意な最小解となるためには、$Q_{uu,n}$が正定値であることが必要である。

$$
Q_{uu,n} \succ 0
$$

対称行列が正定値なら、下記のようにCholesky分解が可能である。

$$
Q_{uu,n} = L_n L_n^T
$$

したがって、Cholesky分解に成功するかどうかによって、正定値性を数値的に確認できる。

```python
try:
    L = np.linalg.cholesky(Q_uu_n)
except np.linalg.LinAlgError:
    raise ValueError(
        f"Q_uu is not positive definite at n={n}"
    )
```

Cholesky分解を用いて対称正定値行列である連立方程式を解くこともできる。SciPyでは、次のように$k_n, K_n$を解くことが出来る。

```python
from scipy.linalg import cho_factor, cho_solve

factor = cho_factor(Q_uu_n, lower=True)

k_n = cho_solve(factor, -Q_u_n)
K_n = cho_solve(factor, -Q_uX_n)
```
Cholesky分解に成功すれば、$Q_{uu,n}$が数値的に正定値であることを確認できる。また、その分解結果を利用して、一般のLU分解よりも効率的に連立方程式を解ける。
ただし、正定値性は数値的な良条件性を保証するものではない。

条件数と正則化については、Step8で扱う。

## 6. backward pass のPython処理

一連の流れは次のようになる。

In [ ]:
from scipy.linalg import cho_factor, cho_solve
import numpy as np

def backward_pass(
        A_array,
        B_array,
        l_X_array,
        l_u_array,
        l_XX_array,
        l_uX_array,
        l_uu_array,
        phi_X,
        phi_XX,
):
    N = A_array.shape[0]
    nx = A_array.shape[1]
    nu = B_array.shape[2]


    k_array = np.zeros((N, nu))
    K_array = np.zeros((N, nu, nx))

    # 終端価値関数
    V_X = phi_X.copy()
    V_XX = phi_XX.copy()
    V_XX = 0.5 * (V_XX + V_XX.T) # 対称化

    # 逆方向に計算, n = N-1 , ... , 0
    for n in reversed(range(N)):
        A_n = A_array[n]
        B_n = B_array[n]

        l_X_n = l_X_array[n]
        l_u_n = l_u_array[n]
        l_XX_n = l_XX_array[n]
        l_uX_n = l_uX_array[n]
        l_uu_n = l_uu_array[n]

        # Q関数の係数
        Q_X = l_X_n + A_n.T @ V_X
        Q_u = l_u_n + B_n.T @ V_X

        Q_uu = l_uu_n + B_n.T @ V_XX @ B_n
        Q_uX = l_uX_n + B_n.T @ V_XX @ A_n
        Q_XX = l_XX_n + A_n.T @ V_XX @ A_n

        # 対称化
        Q_uu = 0.5 * (Q_uu + Q_uu.T)

        # 入力修正項の係数
        try:
            factor = cho_factor(Q_uu, lower=True)
        except np.linalg.LinAlgError:
            raise ValueError(
                f"Q_uu is not positive definite at n={n}"
            )

        k_n = cho_solve(factor, -Q_u)
        K_n = cho_solve(factor, -Q_uX)

        k_array[n] = k_n
        K_array[n] = K_n

        # 価値関数を時刻nへ更新
        V_X = Q_X + Q_uX.T @ k_n
        V_XX = Q_XX + Q_uX.T @ K_n

        # 次のbackward pass へ渡す前に対称化
        V_XX = 0.5 * (V_XX + V_XX.T)

    return k_array, K_array



以上